In [4]:
import mne
file_path = 'C:/Users/Nikita/Desktop/dp/sub-01_task-faces_eeg.bdf'
raw = mne.io.read_raw_bdf(file_path, preload=True)

print(f"Количество каналов: {len(raw.ch_names)}")
print(f"Частота дискретизации: {raw.info['sfreq']} Гц")

print("2. Применяем Notch-фильтр (удаляем 50 Гц)...")
raw.notch_filter(freqs=50.0)
print("3. Применяем Bandpass-фильтр (0.5 - 45 Гц)...")
raw.filter(l_freq=0.5, h_freq=45.0)

print("Готово!")

Extracting BDF parameters from C:/Users/Nikita/Desktop/dp/sub-01_task-faces_eeg.bdf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 2060287  =      0.000 ...  1006.000 secs...
Количество каналов: 129
Частота дискретизации: 2048.0 Гц
2. Применяем Notch-фильтр (удаляем 50 Гц)...
Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 49 - 51 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 49.38
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 49.12 Hz)
- Upper passband edge: 50.62 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 50.88 Hz)
- Filter length: 13517 samples (6.600 s)

3. Применяем Bandpass-фильтр (0.5 - 45 Гц)...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0

In [ ]:
import numpy as np
from statsmodels.tsa.stattools import adfuller
# Снижение частоты
print("Делаем даунсэмплинг до 250 Гц...")
raw.resample(250.0, npad="auto")

channel_data = raw.get_data()[0]

print(f"Количество точек в канале после прореживания: {len(channel_data)}")

# 3. Запускаем ADF-тест (расширенный тест Дики-Фуллера)
print("\nЗапускаем ADF-тест на стационарность...")
adf_result = adfuller(channel_data, autolag="AIC")

# 4. Выводим результаты
print(f"ADF Statistic (Статистика теста): {adf_result[0]:.4f}")
print(f"p-value (p-значение): {adf_result[1]:.4f}")
print("Критический уровень (Critical Values):")
for key, value in adf_result[4].items():
    print(f"   {key}: {value:.4f}")

if adf_result[1] < 0.05:
    print(
        "\nРезультат: p-value < 0.05 -> Ряд стационарный (нулевая гипотеза отвергается)."
    )
else:
    print(
        "\nРезультат: p-value >= 0.05 -> Ряд НЕСТАЦИОНАРНЫЙ (содержит единичный корень)."
    )

Делаем даунсэмплинг до 250 Гц...
Sampling frequency of the instance is already 250.0, returning unmodified.
Количество точек в канале после прореживания: 251500

Запускаем ADF-тест на стационарность...
ADF Statistic (Статистика теста): -45.1419
p-value (p-значение): 0.0000
Критический уровень (Critical Values):
   1%: -3.4304
   5%: -2.8616
   10%: -2.5668

Результат: p-value < 0.05 -> Ряд стационарный (нулевая гипотеза отвергается).


C:\Users\Nikita\AppData\Local\Temp\ipykernel_6252\2607730654.py:13: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_result = adfuller(channel_data, autolag="AIC")


In [ ]:
import pandas as pd
from statsmodels.tsa.api import VAR
data_subset = pd.DataFrame(raw.get_data()[:4].T, columns=[f"Ch_{i}" for i in range(4)])
print(
    f"Размер рабочей матрицы для VAR: {data_subset.shape} (строки-время, столбцы-каналы)"
)
model = VAR(data_subset)

# 3. Подбираем оптимальный порядок лагов по информационному критерию Акаике (AIC)
print("\nПодбираем оптимальное количество лагов...")
lag_order_results = model.select_order(maxlags=10)
print(f"Рекомендуемое количество лагов по AIC: {lag_order_results.aic}")

# Выбираем число лагов (берем то, что порекомендовал AIC, но не меньше 1)
optimal_lags = max(1, lag_order_results.aic)

# 4. Обучаем VAR-модель
print(f"\nОбучаем модель VAR с лагом p={optimal_lags}...")
results = model.fit(optimal_lags)

# Выводим краткую сводку по модели
print(results.summary())

# 5. Проводим тест причинности по Грейнджеру
# Проверим, вызывает ли Канал 0 (causing) Канал 1 (caused)
print("\n--- Тест причинности по Грейнджеру ---")
causality_test = results.test_causality(caused="Ch_1", causing="Ch_0", kind="wald")
print(causality_test.summary())

Размер рабочей матрицы для VAR: (251500, 4) (строки-время, столбцы-каналы)

Подбираем оптимальное количество лагов...
Рекомендуемое количество лагов по AIC: 10

Обучаем модель VAR с лагом p=10...
  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Mon, 26, Oct, 2026
Time:                     15:58:32
--------------------------------------------------------------------
No. of Equations:         4.00000    BIC:                   -136.947
Nobs:                     251490.    HQIC:                  -136.952
Log likelihood:       1.57940e+07    FPE:                3.32550e-60
AIC:                     -136.953    Det(Omega_mle):     3.32333e-60
--------------------------------------------------------------------
Results for equation Ch_0
              coefficient       std. error           t-stat            prob
---------------------------------------------------------------------------
const            0.000000         0.0

In [ ]:
channels = [f"Ch_{i}" for i in range(4)]
p_matrix = pd.DataFrame(np.zeros((4, 4)), index=channels, columns=channels)

# Проходим циклом по всем парам каналов
for causing in channels:
    for caused in channels:
        if causing != caused:
            test_result = results.test_causality(
                caused=caused, causing=causing, kind="wald"
            )
            p_matrix.loc[causing, caused] = test_result.pvalue
        else:
            p_matrix.loc[causing, caused] = np.nan

print("--- Матрица p-value причинности по Грейнджеру ---")
print("(Строки = причиняющий канал (causing), Столбцы = затронутый канал (caused))\n")
print(p_matrix.round(4))

--- Матрица p-value причинности по Грейнджеру ---
(Строки = причиняющий канал (causing), Столбцы = затронутый канал (caused))

      Ch_0  Ch_1  Ch_2  Ch_3
Ch_0   NaN   0.0   0.0   0.0
Ch_1   0.0   NaN   0.0   0.0
Ch_2   0.0   0.0   NaN   0.0
Ch_3   0.0   0.0   0.0   NaN


In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.api import VAR

channels_idx = [0, 32, 64, 96]
channel_names = [f"Ch_{i}" for i in channels_idx]

data_subset_far = pd.DataFrame(
    raw.get_data()[channels_idx].T, columns=channel_names
)

model_far = VAR(data_subset_far)
lag_res = model_far.select_order(maxlags=10)
optimal_lags_far = max(1, lag_res.aic)

results_far = model_far.fit(optimal_lags_far)

p_matrix_far = pd.DataFrame(
    np.zeros((4, 4)), index=channel_names, columns=channel_names
)

for causing in channel_names:
    for caused in channel_names:
        if causing != caused:
            test_res = results_far.test_causality(
                caused=caused, causing=causing, kind="wald"
            )
            p_matrix_far.loc[causing, caused] = test_res.pvalue
        else:
            p_matrix_far.loc[causing, caused] = np.nan

print(p_matrix_far.round(4))

       Ch_0  Ch_32  Ch_64  Ch_96
Ch_0    NaN    0.0    0.0    0.0
Ch_32   0.0    NaN    0.0    0.0
Ch_64   0.0    0.0    NaN    0.0
Ch_96   0.0    0.0    0.0    NaN
